In [1]:
import pandas as pd
import jieba
from gensim.models import Word2Vec

# 1. 读取train.csv，取正确的评论列（第二列comment）
df = pd.read_csv("train.csv")
# 核心修正：取第二列的评论文本，不是第一列的标签
texts = df['comment']

# 2. 中文分词
sentences = []
for content in texts:
    cut_words = list(jieba.cut(str(content)))
    # 过滤空字符、空格、单字
    words = [word for word in cut_words if word.strip() and len(word) > 1]
    sentences.append(words)

# 3. 训练Skip-Gram模型（sg=1，严格符合作业要求）
model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,  # 保留出现≥2次的词，数据量足够，不用设1
    workers=4,
    sg=1  # 1=Skip-Gram，0=CBOW
)
wv = model.wv
print("Skip-Gram 模型训练完成！")
print(f"模型词表总词汇数：{len(wv)}")

C:\Users\11464\.conda\envs\nlp_lyf\lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\11464\AppData\Local\Temp\jieba.cache
Loading model cost 0.617 seconds.
Prefix dict has been built successfully.


Skip-Gram 模型训练完成！
模型词表总词汇数：4742


In [2]:
# 验证是否为Skip-Gram模型
if model.sg == 1:
    print("任务1：当前使用的是 Skip-Gram（跳字模型），符合作业要求")
else:
    print("模型错误，请修改 sg 参数为 1")

任务1：当前使用的是 Skip-Gram（跳字模型），符合作业要求


In [3]:
# 任务2：输出“环境”的词向量和形状
word = "环境"
if word in wv:
    vec = wv[word]
    print(f"【{word}】的词向量：\n{vec}")
    print(f"【{word}】词向量形状（维度）：{vec.shape}")
else:
    print(f"词汇【{word}】不在词表中")

【环境】的词向量：
[ 0.06673237  0.09023923  0.02828225  0.01861262  0.11516884 -0.43417355
  0.4102064   0.46206325 -0.23644277 -0.4573569   0.15655378 -0.51426077
 -0.12583354  0.2634862   0.06970561 -0.20528749  0.07200456 -0.2809265
 -0.11228362 -0.42343387 -0.02861345  0.3625907   0.00378313 -0.030002
  0.02898786 -0.25561553  0.09867178 -0.21429215 -0.24403672 -0.21921313
  0.46103597 -0.24154705  0.1748139  -0.49767625 -0.2599239   0.7602921
  0.3946171  -0.2319755  -0.1408934  -0.52846676  0.36684707 -0.07131778
 -0.25556332 -0.13353005  0.30946302  0.2490916  -0.23057784  0.01363211
  0.29023933  0.0334402   0.07607085 -0.5505886  -0.0339322  -0.05509439
 -0.28212094  0.06652176  0.34561056 -0.12856823 -0.3614445  -0.21931279
  0.19754007 -0.03474994  0.2489452  -0.36442208 -0.54275036  0.5012582
  0.03509679  0.1027981  -0.44032383  0.3538605  -0.30440584  0.2885235
  0.21411961  0.34301868  0.38256642 -0.12994753 -0.38559732  0.20155054
 -0.39463675 -0.18436618 -0.23397824 -0.0501366

In [4]:
# 任务3：查找与“好吃”最接近的3个词
target_word = "好吃"
if target_word in wv:
    similar_words = wv.most_similar(target_word, topn=3)
    print("任务3：与【好吃】语义最接近的3个词语：")
    for word, score in similar_words:
        print(f"词语：{word}，相似度：{score:.4f}")
else:
    print(f"词汇【{target_word}】不在词表中")

任务3：与【好吃】语义最接近的3个词语：
词语：他家，相似度：0.9448
词语：特别，相似度：0.9427
词语：真心，相似度：0.9373


In [5]:
# 任务4：计算相似度
word1 = "好吃"
word2 = "美味"
word3 = "蟑螂"

# 好吃 & 美味
if word1 in wv and word2 in wv:
    sim1 = wv.similarity(word1, word2)
    print(f"任务4：【{word1}】与【{word2}】的相似度：{sim1:.4f}")
else:
    print("词汇【好吃】或【美味】不在词表中")

# 好吃 & 蟑螂
if word1 in wv and word3 in wv:
    sim2 = wv.similarity(word1, word3)
    print(f"任务4：【{word1}】与【{word3}】的相似度：{sim2:.4f}")
else:
    print("词汇【好吃】或【蟑螂】不在词表中")

任务4：【好吃】与【美味】的相似度：0.8982
任务4：【好吃】与【蟑螂】的相似度：0.4043


In [6]:
# 任务5：向量类比运算
positive_list = ["餐厅", "聚会"]
negative_list = ["安静"]

try:
    result = wv.most_similar(positive=positive_list, negative=negative_list, topn=1)
    print("任务5：向量运算【餐厅 + 聚会 - 安静】结果：")
    print(f"最相关词语：{result[0][0]}，匹配度：{result[0][1]:.4f}")
except KeyError:
    print("参与运算的词汇存在缺失，无法完成向量运算")

任务5：向量运算【餐厅 + 聚会 - 安静】结果：
最相关词语：买过，匹配度：0.9716
